# Lesson 3: Code Generation + Sandboxed Execution

## What You'll Learn

1. **Code generation** — LLM writes pandas/matplotlib code from natural language
2. **Sandboxed execution** — Run generated code safely in an isolated process
3. **Error-feedback loop** — When code fails, agent sees the error and self-corrects
4. **Visualization generation** — Agent creates charts and returns image paths
5. **Production safety** — Timeouts, output limits, resource constraints

---

## Why Code Generation > SQL Alone

| Task | SQL | Python/pandas |
|------|-----|---------------|
| Filter and aggregate | Great | Good |
| Statistical tests | Limited | Great |
| Visualization | No | Great |
| Time series analysis | Awkward | Great |
| Custom transformations | Limited | Great |

The tradeoff: **Python code is more powerful but riskier.** LLMs could generate
code that accesses files, contacts the network, or runs forever. That is why we sandbox.

---

## Setup

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import pandas as pd
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelRetry
from analyst.tools.code_executor import execute_code_subprocess, ExecutionResult

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
print(f"Data directory: {DATA_DIR}")
print(f"Sales data: {sales_df.shape}")

---
## Part 1: The Code Executor

Subprocess-based executor for development. For production, use Docker or E2B.

In [ ]:
# Test 1: Basic execution
result = execute_code_subprocess(
    code=(
        "import pandas as pd\n"
        "df = pd.read_csv(f'{DATA_DIR}/sample_sales.csv')\n"
        "print(f'Loaded {len(df)} rows')\n"
        "print(f'Total revenue: ${df.revenue.sum():,.2f}')\n"
    ),
    data_dir=DATA_DIR,
    timeout_seconds=10,
)
print(f"Success: {result.success}")
print(f"Output: {result.stdout}")
print(f"Time: {result.execution_time_ms:.0f}ms")

In [ ]:
# Test 2: Chart generation
chart_code = (
    "import pandas as pd\n"
    "import matplotlib.pyplot as plt\n"
    "df = pd.read_csv(f'{DATA_DIR}/sample_sales.csv')\n"
    "rev = df.groupby('region')['revenue'].sum().sort_values(ascending=False)\n"
    "plt.figure(figsize=(8, 5))\n"
    "rev.plot(kind='bar', color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])\n"
    "plt.title('Total Revenue by Region')\n"
    "plt.ylabel('Revenue ($)')\n"
    "plt.tight_layout()\n"
    "print(rev.to_string())\n"
)
result = execute_code_subprocess(code=chart_code, data_dir=DATA_DIR)
print(f"Success: {result.success}")
print(f"Output: {result.stdout}")
print(f"Generated files: {result.generated_files}")

if result.generated_files:
    from IPython.display import Image, display
    for f in result.generated_files:
        if f.endswith('.png'):
            display(Image(filename=f))

In [ ]:
# Test 3: Code that FAILS — errors are captured cleanly
result = execute_code_subprocess(
    code=(
        "import pandas as pd\n"
        "df = pd.read_csv(f'{DATA_DIR}/sample_sales.csv')\n"
        "print(df['profit'].mean())  # 'profit' column does not exist\n"
    ),
    data_dir=DATA_DIR,
)
print(f"Success: {result.success}")
print(f"Error: {result.stderr[-300:]}")

In [ ]:
# Test 4: Timeout enforcement — long-running code is killed
result = execute_code_subprocess(
    code="import time\nprint('Starting...')\ntime.sleep(60)\nprint('Done!')",
    timeout_seconds=3,
)
print(f"Success: {result.success}")
print(f"Error: {result.stderr}")
print(f"Time: {result.execution_time_ms:.0f}ms")

### Sandbox comparison

| Property | Subprocess (dev) | Docker (prod) | E2B (cloud) |
|----------|-----------------|---------------|-------------|
| Timeout | Yes | Yes | Yes |
| Memory limit | No | Yes | Yes |
| Network isolation | No | Yes | Yes |
| Filesystem isolation | No | Yes | Yes |
| Cost | Free | Free | ~$0.01/run |

---

## Part 2: LLM-Powered Code Generation Agent

In [ ]:
@dataclass
class CodeAgentDeps:
    """Runtime dependencies for the code agent."""
    data_dir: str
    available_files: list[str] = field(default_factory=list)
    timeout_seconds: int = 30


class CodeAnalysisResult(BaseModel):
    """Structured result from code-based analysis."""
    answer: str = Field(description="Natural language answer")
    code: str = Field(description="Python code that produced the answer")
    raw_output: str = Field(description="Raw stdout from execution")
    charts_generated: bool = Field(description="Whether charts were created")
    confidence: float = Field(ge=0.0, le=1.0)


code_agent = Agent(
    "openai:local-model",
    deps_type=CodeAgentDeps,
    output_type=CodeAnalysisResult,
    system_prompt=(
        "You are a Python data analyst. Write pandas code to answer questions.\n\n"
        "RULES:\n"
        "1. Always use the run_python tool to execute code\n"
        "2. Load data using: pd.read_csv(f'{DATA_DIR}/filename.csv')\n"
        "3. Always print() your results\n"
        "4. For charts, use matplotlib (auto-saved)\n"
        "5. If code fails, read the error and fix it\n"
        "6. Keep code concise"
    ),
    retries=3,
)


@code_agent.system_prompt
def add_data_context(ctx: RunContext[CodeAgentDeps]) -> str:
    lines = [f"\nData directory: {ctx.deps.data_dir}", "Available files:"]
    for fname in ctx.deps.available_files:
        fpath = Path(ctx.deps.data_dir) / fname
        if fpath.exists() and fpath.suffix == '.csv':
            df = pd.read_csv(fpath)
            cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
            lines.append(f"  - {fname}: {len(df)} rows | Columns: {cols}")
            lines.append(f"    Sample row: {df.iloc[0].to_dict()}")
    return "\n".join(lines)


@code_agent.tool
def run_python(ctx: RunContext[CodeAgentDeps], code: str) -> str:
    """Execute Python code. pandas/matplotlib/numpy available.
    Use DATA_DIR for data files. Always print() results."""
    result = execute_code_subprocess(
        code=code,
        data_dir=ctx.deps.data_dir,
        timeout_seconds=ctx.deps.timeout_seconds,
    )
    if not result.success:
        raise ModelRetry(
            f"Code failed. Fix and retry.\nError:\n{result.stderr}\nCode:\n{code}"
        )
    parts = []
    if result.stdout:
        parts.append(f"Output:\n{result.stdout}")
    if result.generated_files:
        parts.append(f"Files: {result.generated_files}")
    parts.append(f"Time: {result.execution_time_ms:.0f}ms")
    return "\n".join(parts) if parts else "Executed (no output)."


print("Code agent ready.")

In [ ]:
# Run 1: Simple aggregation
deps = CodeAgentDeps(
    data_dir=DATA_DIR,
    available_files=["sample_sales.csv", "sample_employees.csv"],
)

result = code_agent.run_sync("What are the top 3 products by total revenue?", deps=deps)
print(f"Answer: {result.output.answer}\n")
print(f"Code:\n{result.output.code}\n")
print(f"Confidence: {result.output.confidence:.0%}")

In [ ]:
# Run 2: Chart generation
result = code_agent.run_sync(
    "Create a bar chart: total revenue by category (Electronics vs Home), "
    "also show profit margin for each.",
    deps=deps,
)
print(f"Answer: {result.output.answer}")
print(f"Charts: {result.output.charts_generated}")

if result.output.charts_generated:
    from IPython.display import Image, display
    import tempfile
    for p in sorted(Path(tempfile.gettempdir()).rglob("chart_*.png")):
        display(Image(filename=str(p)))
        break

In [ ]:
# Run 3: Complex multi-step
result = code_agent.run_sync(
    "Analyze weekly revenue trends: "
    "1. Week-over-week growth rate "
    "2. Highest/lowest growth weeks "
    "3. Any seasonal pattern?",
    deps=deps,
)
print(f"Answer:\n{result.output.answer}")
print(f"\nRequests: {result.usage().requests} | Confidence: {result.output.confidence:.0%}")

---
## Part 3: Tracing the Error-Feedback Loop

In [ ]:
result = code_agent.run_sync(
    "Calculate Pearson correlation between unit_price and quantity. "
    "Is there a negative correlation?",
    deps=deps,
)

print("=== MESSAGE FLOW ===")
tool_calls = 0
for msg in result.all_messages():
    for part in msg.parts:
        kind = getattr(part, 'part_kind', 'unknown')
        if kind == 'tool-call':
            tool_calls += 1
            print(f"\n[CALL #{tool_calls}] {getattr(part, 'tool_name', '?')}")
            print(f"  {str(part.args)[:200]}")
        elif kind == 'tool-return':
            print(f"[RESULT] {str(part.content)[:200]}")
        elif kind == 'retry-prompt':
            print(f"[RETRY] {str(part.content)[:200]}")

print(f"\n=== {tool_calls} tool calls ===")
print(f"Answer: {result.output.answer}")

---
## Part 4: Static Code Validation

Before running code, static analysis catches dangerous patterns.
See `src/analyst/tools/code_validator.py` for the full implementation.

The validator uses Python's `ast` module to walk the syntax tree and flag:
- Banned imports (network libraries, subprocess, etc.)
- Dangerous function calls (dynamic code execution)
- Dangerous method calls (system commands, file deletion)

This is a **first line of defense** — the sandbox provides real isolation.

In [ ]:
import ast

BANNED_IMPORTS = {"requests", "urllib", "socket", "http", "subprocess", "shutil"}
BANNED_CALLS = {"compile", "__import__"}


def validate_code_safety(code: str) -> tuple[bool, list[str]]:
    """Static analysis for dangerous patterns. Returns (is_safe, issues)."""
    issues = []
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return False, [f"Syntax error: {e}"]

    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            mod = None
            if isinstance(node, ast.Import):
                mod = node.names[0].name.split(".")[0]
            elif node.module:
                mod = node.module.split(".")[0]
            if mod and mod in BANNED_IMPORTS:
                issues.append(f"Banned import: {mod}")

        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id in BANNED_CALLS:
                issues.append(f"Banned call: {node.func.id}()")

    return len(issues) == 0, issues


# Quick test
safe_code = "import pandas as pd\nprint(pd.read_csv('data.csv').head())"
unsafe_code = "import requests\nrequests.get('http://example.com')"

print(f"Safe code:   {validate_code_safety(safe_code)}")
print(f"Unsafe code: {validate_code_safety(unsafe_code)}")

In [ ]:
# Enhanced agent with safety validation
safe_agent = Agent(
    "openai:local-model",
    deps_type=CodeAgentDeps,
    output_type=CodeAnalysisResult,
    system_prompt=(
        "You are a Python data analyst. Write pandas code to answer questions.\n"
        "Only use pandas, numpy, matplotlib, scipy. No network or system libraries."
    ),
    retries=3,
)

@safe_agent.system_prompt
def data_info(ctx: RunContext[CodeAgentDeps]) -> str:
    lines = [f"\nData dir: {ctx.deps.data_dir}", "Files:"]
    for fname in ctx.deps.available_files:
        fpath = Path(ctx.deps.data_dir) / fname
        if fpath.exists() and fpath.suffix == '.csv':
            df = pd.read_csv(fpath)
            cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
            lines.append(f"  - {fname}: {len(df)} rows | {cols}")
    return "\n".join(lines)

@safe_agent.tool
def run_validated_python(ctx: RunContext[CodeAgentDeps], code: str) -> str:
    """Execute validated Python code. Only safe libraries allowed."""
    is_safe, issues = validate_code_safety(code)
    if not is_safe:
        raise ModelRetry(
            "Code blocked by safety check:\n"
            + "\n".join(f"  - {i}" for i in issues)
            + "\nUse only pandas, numpy, matplotlib."
        )
    result = execute_code_subprocess(
        code=code, data_dir=ctx.deps.data_dir,
        timeout_seconds=ctx.deps.timeout_seconds,
    )
    if not result.success:
        raise ModelRetry(f"Code failed:\n{result.stderr}\nFix and retry.")

    parts = []
    if result.stdout:
        stdout = result.stdout[:3000]
        if len(result.stdout) > 3000:
            stdout += "\n...(truncated)"
        parts.append(f"Output:\n{stdout}")
    if result.generated_files:
        parts.append(f"Files: {result.generated_files}")
    return "\n".join(parts) or "Done (no output)."


result = safe_agent.run_sync(
    "Average profit margin per product as a percentage.", deps=deps,
)
print(f"Answer: {result.output.answer}\nCode:\n{result.output.code}")

---
## Part 5: Batch Testing

In [ ]:
questions = [
    "Total revenue across all products?",
    "Which region has the highest average profit margin?",
    "Create a line chart of weekly revenue trends.",
    "Correlation between quantity and unit_price?",
]

for i, q in enumerate(questions, 1):
    print(f"\n{'='*50}\nQ{i}: {q}\n{'='*50}")
    try:
        r = safe_agent.run_sync(q, deps=deps)
        print(f"A: {r.output.answer[:250]}")
        u = r.usage()
        print(f"Confidence: {r.output.confidence:.0%} | Tokens: {u.input_tokens+u.output_tokens}")
    except Exception as e:
        print(f"FAILED: {e}")

---
## Summary

| Component | Purpose |
|-----------|----------|
| Subprocess executor | Isolated code execution for dev |
| Docker sandbox (in `src/`) | Full production isolation |
| AST-based validator | Pre-execution safety gate |
| ModelRetry on failure | Self-correcting code gen |
| Output truncation | Prevent context overflow |
| Chart auto-capture | Visual analysis support |

### Key production patterns
1. Always sandbox LLM-generated code
2. Defense in depth: static validation + process isolation
3. Enforce timeouts on all executions
4. Truncate large outputs
5. Errors are feedback signals for the LLM to self-correct

**Next: Lesson 4 — ReAct reasoning loop**

---
## Exercises

1. Build and test `Dockerfile.sandbox`, compare with subprocess
2. Extend the AST validator to detect infinite loops and memory bombs
3. Build a code-reviewer agent that checks generated code before execution
4. Build a visualization-specialist agent focused only on chart quality